# Lesson 04: Support Vector Machines and Kernels - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: maximum margin, kernels, and HD-LSS classification.

## Learning goals
1. Generate a nonlinear pre-failure classification problem.
2. Fit linear and RBF SVM models.
3. Compare decision boundaries and support vector counts.
4. Explain the role of C and gamma.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
n = 140
voltage_drop = rng.normal(0, 1, n)
temperature_rise = rng.normal(0, 1, n)
X = np.column_stack([voltage_drop, temperature_rise])
y = ((voltage_drop**2 + 0.7*temperature_rise**2 + 0.4*voltage_drop*temperature_rise) > 1.4).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=1)
models = {
    "linear": make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)),
    "rbf": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=5.0, gamma=0.8)),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    svc = model.named_steps["svc"]
    print(f"{name:6s} accuracy={accuracy_score(y_test, pred):.3f}, recall={recall_score(y_test, pred):.3f}, support_vectors={svc.n_support_.sum()}")

rbf = models["rbf"]
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 220),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 220))
Z = rbf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.2, levels=[-0.5,0.5,1.5], colors=["tab:blue", "tab:orange"])
plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", edgecolor="k")
plt.xlabel("Voltage drop feature")
plt.ylabel("Temperature rise feature")
plt.title("RBF SVM boundary")
plt.grid(alpha=0.25)
plt.show()

print("C controls margin violations. gamma controls how local the RBF kernel is.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?